In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

In [2]:
# 初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

In [3]:
config = {"configurable": {"thread_id": "test_day08"}}
# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)
print(result["messages"][-1].content)
print("======================================")
# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)

嗯，我记住了，小明。你今年 **30岁**。😊

不过你已经跟我说了好几次同样的话啦——是有什么特别的原因吗？还是想测试我记不记得住？😄 有什么需要帮忙的，尽管说～
你今年 **30岁**，小明。😊

这个问题你已经问过好几次啦～我都记住啦，不会忘的。😄

要不要聊点别的？比如今天过得怎么样，或者有什么想做的事？


In [7]:
from langchain.messages import HumanMessage, AIMessage
# 查看最新的对话的快照
snap_shot = await agent.aget_state(config)
print(snap_shot)
print("===============")
messages =  snap_shot.values["messages"]
# print(messages)
for message in messages:
    # message.pretty_print()
    if isinstance(message, HumanMessage):
        role="user"
    elif isinstance(message, AIMessage):
        role="assistant"
    else:
        continue
    print({"role": role, "content": message.content})


StateSnapshot(values={'messages': [HumanMessage(content='我叫小明，今年30岁。', additional_kwargs={}, response_metadata={}, id='323904b7-4cc1-4936-be59-1c65ad14891c'), AIMessage(content='你好，小明！很高兴认识你。30岁正是人生中精力充沛、充满可能性的阶段呢。\n\n作为AI助手，我可以陪你聊聊天、解答问题、提供建议，或者帮你处理一些文字、信息类的任务。有什么想聊的，或者需要我帮忙的吗？无论是工作、生活、兴趣爱好，还是单纯想找人说说话，我都在这里。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 11, 'total_tokens': 89, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '88c644b9-92b3-4391-a694-e79a870b00ba', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09385-1452-73e0-99ba-a7a3bdc749cf-0', tool_calls=[], invalid_tool_calls=[], usage_met

In [8]:
thread_id = config["configurable"]["thread_id"]
await checkpointer.adelete_thread(thread_id)
snapshot = await agent.aget_state(config)
print(snapshot.values)

{}


In [2]:
# 初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
# 创建Agent
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

In [3]:
config = {"configurable": {"thread_id": "test_day08"}}
# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)
print(result["messages"][-1].content)
print("======================================")
# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)

你好，小明！30岁真是很好的年纪呢。😊

有什么我可以帮你的吗？不管是工作、生活、学习上的问题，还是只是想聊聊天，我都很乐意陪你聊聊。
你今年 **30岁** 呀，小明。😊

你刚刚告诉我：“我叫小明，今年30岁。”所以我就记住啦。


In [4]:
# 查看最新的对话的快照
snap_shot = await agent.aget_state(config)
print(snap_shot)

StateSnapshot(values={'messages': [HumanMessage(content='我叫小明，今年30岁。', additional_kwargs={}, response_metadata={}, id='e47ed391-8b34-490a-827b-febe5007bbb6'), AIMessage(content='你好，小明！30岁真是很好的年纪呢。😊\n\n有什么我可以帮你的吗？不管是工作、生活、学习上的问题，还是只是想聊聊天，我都很乐意陪你聊聊。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 11, 'total_tokens': 51, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '7318093d-b8cf-4839-98d9-934da5ba6001', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09379-f2d2-7d50-95b7-e8ca29434e85-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 40, 'total_tokens': 

In [1]:
messages =  snap_shot.values["messages"]
from langchain.messages import HumanMessage, AIMessage
# print(messages)
for message in messages:
    # message.pretty_print()
    if isinstance(message, HumanMessage):
        role="user"
    elif isinstance(message, AIMessage):
        role="assistant"
    else:
        continue
    print({"role": role, "content": message.content})



NameError: name 'snap_shot' is not defined